# Sanity Check - Step 06: Epoching

Überprüft:
- Epochen korrekt aus Rohdaten extrahiert
- Ereignis-Anzahl und IDs
- Epoche-Parameter (tmin, tmax)
- Baseline-Fenster

In [ ]:
import sys
import mne
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent / 'eeg_pipeline'))
import config

print("Setup erfolgreich")

In [ ]:
subject_id = config.SUBJECTS[0]

# Step 05 Output laden (VORHER - Downsampled Raw)
p1_path_before = config.OUTPUT_DIR / f"sub-{subject_id}_P1_downsampled.fif"
p2_path_before = config.OUTPUT_DIR / f"sub-{subject_id}_P2_downsampled.fif"

raw_p1_before = mne.io.read_raw_fif(str(p1_path_before), preload=False)
raw_p2_before = mne.io.read_raw_fif(str(p2_path_before), preload=False)

print(f"\n=== VORHER (Step 05 Output - Downsampled) ===\n")
print(f"Person 1:")
print(f"  EEG Kanäle: {len(mne.pick_types(raw_p1_before.info, eeg=True))}")
print(f"  Sampling Rate: {raw_p1_before.info['sfreq']:.0f} Hz")
print(f"  Datenlänge: {raw_p1_before.n_times} Samples")
print(f"  Dauer: {raw_p1_before.times[-1]:.2f} Sekunden")

# Extrahiere Events
events_before_p1, event_id_before_p1 = mne.events_from_annotations(raw_p1_before)
print(f"  Ereignisse: {len(events_before_p1)}")
print(f"  Ereignis-IDs: {event_id_before_p1}")

print(f"\nPerson 2:")
print(f"  EEG Kanäle: {len(mne.pick_types(raw_p2_before.info, eeg=True))}")
print(f"  Sampling Rate: {raw_p2_before.info['sfreq']:.0f} Hz")
print(f"  Datenlänge: {raw_p2_before.n_times} Samples")
print(f"  Dauer: {raw_p2_before.times[-1]:.2f} Sekunden")

events_before_p2, event_id_before_p2 = mne.events_from_annotations(raw_p2_before)
print(f"  Ereignisse: {len(events_before_p2)}")
print(f"  Ereignis-IDs: {event_id_before_p2}")

In [ ]:
# Step 06 Output laden (NACHHER - Epoched)
p1_path_after = config.OUTPUT_DIR / f"sub-{subject_id}_P1_epoch.fif"
p2_path_after = config.OUTPUT_DIR / f"sub-{subject_id}_P2_epoch.fif"

epochs_p1_after = mne.read_epochs(str(p1_path_after), preload=True)
epochs_p2_after = mne.read_epochs(str(p2_path_after), preload=True)

print(f"\n=== NACHHER (Step 06 Output - Epoched) ===\n")
print(f"Person 1:")
print(f"  Anzahl Epochen: {len(epochs_p1_after)}")
print(f"  EEG Kanäle: {len(mne.pick_types(epochs_p1_after.info, eeg=True))}")
print(f"  Sampling Rate: {epochs_p1_after.info['sfreq']:.0f} Hz")
print(f"  Epoch Länge: {epochs_p1_after.times[-1] - epochs_p1_after.times[0]:.3f} Sekunden")
print(f"  Epoch Samples: {len(epochs_p1_after.times)}")
print(f"  Event IDs: {epochs_p1_after.event_id}")

print(f"\nPerson 2:")
print(f"  Anzahl Epochen: {len(epochs_p2_after)}")
print(f"  EEG Kanäle: {len(mne.pick_types(epochs_p2_after.info, eeg=True))}")
print(f"  Sampling Rate: {epochs_p2_after.info['sfreq']:.0f} Hz")
print(f"  Epoch Länge: {epochs_p2_after.times[-1] - epochs_p2_after.times[0]:.3f} Sekunden")
print(f"  Epoch Samples: {len(epochs_p2_after.times)}")
print(f"  Event IDs: {epochs_p2_after.event_id}")

In [ ]:
print(f"\n=== EPOCHING PARAMETER VALIDIERUNG ===\n")

# Check ob die Epochen-Parameter mit config übereinstimmen
expected_tmin = config.EPOCH_TMIN
expected_tmax = config.EPOCH_TMAX
expected_epoch_length = expected_tmax - expected_tmin

actual_epoch_length_p1 = epochs_p1_after.times[-1] - epochs_p1_after.times[0]
actual_epoch_length_p2 = epochs_p2_after.times[-1] - epochs_p2_after.times[0]

print(f"Konfigurierte Parameter (config.py):")
print(f"  EPOCH_TMIN: {expected_tmin} s")
print(f"  EPOCH_TMAX: {expected_tmax} s")
print(f"  Erwartete Epoch-Länge: {expected_epoch_length} s")
print(f"  EPOCH_BASELINE: [{config.EPOCH_BASELINE_MIN}, {config.EPOCH_BASELINE_MAX}] s")

print(f"\nAktuelle Epochen (Person 1):")
print(f"  Epoch Länge: {actual_epoch_length_p1:.3f} s")
print(f"  tmin: {epochs_p1_after.times[0]:.3f} s")
print(f"  tmax: {epochs_p1_after.times[-1]:.3f} s")
print(f"  tmin stimmt überein: {abs(epochs_p1_after.times[0] - expected_tmin) < 0.001}")
print(f"  tmax stimmt überein: {abs(epochs_p1_after.times[-1] - expected_tmax) < 0.001}")

In [ ]:
# Ereignis-Statistik
print(f"\n=== EREIGNIS STATISTIK ===\n")

print(f"Person 1:")
print(f"  Rohdata Ereignisse: {len(events_before_p1)}")
print(f"  Epochen erstellt: {len(epochs_p1_after)}")
print(f"  Epochen pro Ereignis-Typ:")
for event_type, event_id in epochs_p1_after.event_id.items():
    n_epochs = (epochs_p1_after.events[:, 2] == event_id).sum()
    print(f"    {event_type}: {n_epochs} Epochen")

print(f"\nPerson 2:")
print(f"  Rohdata Ereignisse: {len(events_before_p2)}")
print(f"  Epochen erstellt: {len(epochs_p2_after)}")
print(f"  Epochen pro Ereignis-Typ:")
for event_type, event_id in epochs_p2_after.event_id.items():
    n_epochs = (epochs_p2_after.events[:, 2] == event_id).sum()
    print(f"    {event_type}: {n_epochs} Epochen")

In [ ]:
# Vergleiche Durchschnitt Epoch pro Ereignis
print(f"\n=== EPOCH-QUALITÄT ===\n")

data_epochs_p1 = epochs_p1_after.get_data()
data_epochs_p2 = epochs_p2_after.get_data()

print(f"Person 1 Epoch-Daten:")
print(f"  Shape: {data_epochs_p1.shape} (Epochen, Kanäle, Zeitpunkte)")
print(f"  Min: {np.min(data_epochs_p1):.6f} µV")
print(f"  Max: {np.max(data_epochs_p1):.6f} µV")
print(f"  Mean: {np.mean(data_epochs_p1):.6f} µV")
print(f"  Std: {np.std(data_epochs_p1):.6f} µV")
print(f"  NaN Werte: {np.isnan(data_epochs_p1).sum()}")
print(f"  Inf Werte: {np.isinf(data_epochs_p1).sum()}")

print(f"\nPerson 2 Epoch-Daten:")
print(f"  Shape: {data_epochs_p2.shape}")
print(f"  Min: {np.min(data_epochs_p2):.6f} µV")
print(f"  Max: {np.max(data_epochs_p2):.6f} µV")
print(f"  Mean: {np.mean(data_epochs_p2):.6f} µV")
print(f"  Std: {np.std(data_epochs_p2):.6f} µV")
print(f"  NaN Werte: {np.isnan(data_epochs_p2).sum()}")
print(f"  Inf Werte: {np.isinf(data_epochs_p2).sum()}")

In [ ]:
# Visualisiere durchschnittliche Epochen
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Person 1 - Durchschnitt über alle Epochen
ax1 = axes[0]
avg_epoch_p1 = epochs_p1_after.average()
avg_epoch_p1.plot(axes=ax1, show=False)
ax1.set_title('Person 1 - Durchschnittliche Epoche (über alle Ereignisse)')
ax1.set_xlabel('Zeit (s)')
ax1.set_ylabel('Amplitude (µV)')

# Person 2
ax2 = axes[1]
avg_epoch_p2 = epochs_p2_after.average()
avg_epoch_p2.plot(axes=ax2, show=False)
ax2.set_title('Person 2 - Durchschnittliche Epoche (über alle Ereignisse)')
ax2.set_xlabel('Zeit (s)')
ax2.set_ylabel('Amplitude (µV)')

plt.tight_layout()
plt.show()

In [ ]:
# Visualisiere einzelne Epochen aus Person 1
fig = epochs_p1_after[:5].plot(show=False, block=False)  # Erste 5 Epochen
fig.suptitle('Person 1 - Erste 5 Epochen')
plt.show()

In [ ]:
# Topomap der durchschnittlichen Aktivität während Epoche
fig = avg_epoch_p1.plot_topomap(times=[0, 1, 2, 3], show=False)
fig.suptitle('Person 1 - Topomaps zu verschiedenen Zeitpunkten')
plt.show()

In [ ]:
print(f"\n=== SANITY CHECK ZUSAMMENFASSUNG ===\n")

checks = []

# 1. Epochen sollten erstellt sein
epochs_created = len(epochs_p1_after) > 0 and len(epochs_p2_after) > 0
checks.append(("Epochen erstellt", epochs_created))

# 2. Epoch-Länge sollte mit config übereinstimmen
correct_epoch_length = (
    abs(actual_epoch_length_p1 - expected_epoch_length) < 0.01 and
    abs(actual_epoch_length_p2 - expected_epoch_length) < 0.01
)
checks.append((f"Epoch-Länge korrekt ({expected_epoch_length}s)", correct_epoch_length))

# 3. tmin sollte korrekt sein
correct_tmin = (
    abs(epochs_p1_after.times[0] - expected_tmin) < 0.001 and
    abs(epochs_p2_after.times[0] - expected_tmin) < 0.001
)
checks.append((f"tmin korrekt ({expected_tmin}s)", correct_tmin))

# 4. tmax sollte korrekt sein
correct_tmax = (
    abs(epochs_p1_after.times[-1] - expected_tmax) < 0.001 and
    abs(epochs_p2_after.times[-1] - expected_tmax) < 0.001
)
checks.append((f"tmax korrekt ({expected_tmax}s)", correct_tmax))

# 5. Kanal-Namen sollten gleich sein
same_channels = raw_p1_before.ch_names == epochs_p1_after.ch_names
checks.append(("Kanal-Namen gleich", same_channels))

# 6. Keine NaN oder Inf
no_nan_inf = not (np.isnan(data_epochs_p1).any() or np.isinf(data_epochs_p1).any() or
                   np.isnan(data_epochs_p2).any() or np.isinf(data_epochs_p2).any())
checks.append(("Keine NaN/Inf Werte", no_nan_inf))

# 7. Sampling Rate sollte gleich sein
same_sfreq = epochs_p1_after.info['sfreq'] == raw_p1_before.info['sfreq']
checks.append(("Sampling Rate gleich", same_sfreq))

# 8. Epochen sollten nicht zu viele sein (max. config.MAX_EPOCHS)
reasonable_epochs = len(epochs_p1_after) <= config.MAX_EPOCHS and len(epochs_p2_after) <= config.MAX_EPOCHS
checks.append((f"Epochen-Anzahl vernünftig (≤ {config.MAX_EPOCHS})", reasonable_epochs))

for check_name, result in checks:
    status = "✓ PASS" if result else "✗ FAIL"
    print(f"{status}: {check_name}")

all_pass = all(result for _, result in checks)
print(f"\n{'='*50}")
if all_pass:
    print("✓ ALLE CHECKS BESTANDEN")
else:
    print("✗ EINIGE CHECKS FEHLGESCHLAGEN")
print(f"{'='*50}")